In [1]:
!pip install ultralytics opencv-python

In [2]:
import cv2
import time
from collections import defaultdict
from ultralytics import YOLO
import numpy as np
import os

In [16]:
try:
    from google.colab import drive
    drive.mount('/content/drive')

    PROJECT_PATH = "/content/drive/MyDrive/Kerja Praktik Internal/"
    os.chdir(PROJECT_PATH)
    print(f"Project path: {os.getcwd()}")

except ImportError:
    print("Bukan di lingkungan Google Colab, menggunakan path lokal.")
    PROJECT_PATH = "."

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project path: /content/drive/MyDrive/Kerja Praktik Internal


In [17]:

YOLO_MODEL_PATH = 'Human_Detection_Model.pt'
GESTURE_MODEL_PATH = '/FATHAN_ASSISTANCE GESTURE ALERT_VER10/gesture_detection_yolov11n_VER10.pt'
INPUT_VIDEO_PATH = 'sample1.mp4'
OUTPUT_VIDEO_PATH = 'sample1.mp4'


In [18]:
LOITER_THRESHOLD_SECONDS = 5.0

INSTANT_HELP_GESTURES = ["raising-hand"]

def detect_gestures(frame, person_bbox, gesture_model=None):
    if 'track_id' in person_bbox:
        if person_bbox['track_id'] == 2 and np.random.rand() < 0.5:
            return "raising-hand"
    return "berdiri_normal"

def process_video(model, gesture_model, video_path, output_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Tidak bisa membuka video {video_path}")
        return

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

    customer_tracker = defaultdict(lambda: {
        'last_pos': None,
        'loiter_start_time': None,
        'current_gesture': None,
        'gesture_start_time': None,
        'needs_help': False
    })

    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        video_timestamp = frame_count / fps

        results = model.track(frame, persist=True, classes=[0], conf=0.5, verbose=False)

        if results[0].boxes.id is None:
            out.write(frame)
            continue

        detected_boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        track_ids = results[0].boxes.id.cpu().numpy().astype(int)

        for bbox, track_id in zip(detected_boxes, track_ids):
            state = customer_tracker[track_id]
            center_x = (bbox[0] + bbox[2]) // 2
            center_y = (bbox[1] + bbox[3]) // 2
            current_pos = (center_x, center_y)

            # Deteksi diam
            if state['last_pos'] is not None:
                distance = np.sqrt((current_pos[0] - state['last_pos'][0])**2 +
                                   (current_pos[1] - state['last_pos'][1])**2)
                if distance < 20:
                    if state['loiter_start_time'] is None:
                        state['loiter_start_time'] = video_timestamp
                else:
                    state['loiter_start_time'] = None
            else:
                state['loiter_start_time'] = video_timestamp
            state['last_pos'] = current_pos

            # Deteksi gesture
            person_info_for_gesture = {'track_id': track_id}
            detected_gesture = detect_gestures(frame, person_info_for_gesture, gesture_model)

            if detected_gesture == state['current_gesture']:
                if state['gesture_start_time'] is None:
                    state['gesture_start_time'] = video_timestamp
            else:
                state['current_gesture'] = detected_gesture
                state['gesture_start_time'] = video_timestamp

            # Aturan bantuan
            if not state['needs_help']:
              # Raising hand → langsung butuh bantuan
              if state['current_gesture'] in INSTANT_HELP_GESTURES:
                  state['needs_help'] = True
                  print(f"\nINFO: Customer ID {track_id} butuh bantuan (GESTUR INSTAN: {state['current_gesture']}).")

              # Diam lebih dari 5 detik
              elif state['loiter_start_time'] and (video_timestamp - state['loiter_start_time'] > LOITER_THRESHOLD_SECONDS):
                  state['needs_help'] = True
                  print(f"\nINFO: Customer ID {track_id} butuh bantuan (diam terlalu lama).")


        # Gambar kotak & label
        for track_id, state in customer_tracker.items():
          try:
              idx = np.where(track_ids == track_id)[0][0]
              bbox = detected_boxes[idx]
          except IndexError:
              continue

          x1, y1, x2, y2 = bbox
          label = f"ID: {track_id}"
          if state['needs_help']:
              color = (0, 0, 255)  # merah
              label += " - BUTUH BANTUAN"
          elif state['current_gesture'] in INSTANT_HELP_GESTURES:
              color = (0, 165, 255)  # oranye
              label += f" | {state['current_gesture']}"
          elif state['loiter_start_time']:
              color = (0, 255, 255)  # kuning
              label += f" | Diam: {video_timestamp - state['loiter_start_time']:.1f}s"
          else:
              color = (0, 255, 0)  # hijau

          cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
          cv2.putText(frame, label, (x1, y1 - 10),
                      cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        out.write(frame)
        print(f"Processing frame {frame_count} | Video time: {video_timestamp:.2f}s", end='\r')

    cap.release()
    out.release()
    print(f"\nProses selesai. Video disimpan di: {output_path}")


In [19]:
if __name__ == "__main__":
    print("Memuat model YOLO...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    print("Model YOLO berhasil dimuat.")

    INPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample1.mp4'
    OUTPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample1_output.mp4'
    gesture_model = None

    process_video(yolo_model, gesture_model, INPUT_VIDEO_PATH, OUTPUT_VIDEO_PATH)

Memuat model YOLO...
Model YOLO berhasil dimuat.
Processing frame 73 | Video time: 3.04s
INFO: Customer ID 2 butuh bantuan (GESTUR INSTAN: raising-hand).

Proses selesai. Video disimpan di: TES_ASSISTANCE GESTURE ALERT/sample1_output.mp4


In [7]:
if __name__ == "__main__":
    print("Memuat model YOLO...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    print("Model YOLO berhasil dimuat.")

    INPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample2.mp4'
    OUTPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample2_output.mp4'
    gesture_model = None

    process_video(yolo_model, gesture_model, INPUT_VIDEO_PATH, OUTPUT_VIDEO_PATH)

Memuat model YOLO...
Model YOLO berhasil dimuat.
Processing frame 192 | Video time: 8.00s
Proses selesai. Video disimpan di: TES_ASSISTANCE GESTURE ALERT/sample2_output.mp4


In [8]:
if __name__ == "__main__":
    print("Memuat model YOLO...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    print("Model YOLO berhasil dimuat.")

    INPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample3.mp4'
    OUTPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample3_output.mp4'
    gesture_model = None

    process_video(yolo_model, gesture_model, INPUT_VIDEO_PATH, OUTPUT_VIDEO_PATH)

Memuat model YOLO...
Model YOLO berhasil dimuat.

INFO: Customer ID 1 butuh bantuan (diam terlalu lama).

INFO: Customer ID 2 butuh bantuan (GESTUR INSTAN: raising-hand).
Processing frame 192 | Video time: 8.00s
Proses selesai. Video disimpan di: TES_ASSISTANCE GESTURE ALERT/sample3_output.mp4


In [9]:
if __name__ == "__main__":
    print("Memuat model YOLO...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    print("Model YOLO berhasil dimuat.")

    INPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample4.mp4'
    OUTPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample4_output.mp4'
    gesture_model = None

    process_video(yolo_model, gesture_model, INPUT_VIDEO_PATH, OUTPUT_VIDEO_PATH)

Memuat model YOLO...
Model YOLO berhasil dimuat.
Processing frame 20 | Video time: 0.83s
INFO: Customer ID 2 butuh bantuan (GESTUR INSTAN: raising-hand).

INFO: Customer ID 1 butuh bantuan (diam terlalu lama).
Processing frame 192 | Video time: 8.00s
Proses selesai. Video disimpan di: TES_ASSISTANCE GESTURE ALERT/sample4_output.mp4


In [10]:
if __name__ == "__main__":
    print("Memuat model YOLO...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    print("Model YOLO berhasil dimuat.")

    INPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample5.mp4'
    OUTPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample5_output.mp4'
    gesture_model = None

    process_video(yolo_model, gesture_model, INPUT_VIDEO_PATH, OUTPUT_VIDEO_PATH)

Memuat model YOLO...
Model YOLO berhasil dimuat.
Processing frame 121 | Video time: 5.04s
INFO: Customer ID 1 butuh bantuan (diam terlalu lama).

Proses selesai. Video disimpan di: TES_ASSISTANCE GESTURE ALERT/sample5_output.mp4


In [11]:
if __name__ == "__main__":
    print("Memuat model YOLO...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    print("Model YOLO berhasil dimuat.")

    INPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample6.mp4'
    OUTPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample6_output.mp4'
    gesture_model = None

    process_video(yolo_model, gesture_model, INPUT_VIDEO_PATH, OUTPUT_VIDEO_PATH)

Memuat model YOLO...
Model YOLO berhasil dimuat.
Processing frame 1 | Video time: 0.04s
INFO: Customer ID 2 butuh bantuan (GESTUR INSTAN: raising-hand).

INFO: Customer ID 1 butuh bantuan (diam terlalu lama).
Processing frame 192 | Video time: 8.00s
Proses selesai. Video disimpan di: TES_ASSISTANCE GESTURE ALERT/sample6_output.mp4


In [12]:
if __name__ == "__main__":
    print("Memuat model YOLO...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    print("Model YOLO berhasil dimuat.")

    INPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample7.mp4'
    OUTPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample7_output.mp4'
    gesture_model = None

    process_video(yolo_model, gesture_model, INPUT_VIDEO_PATH, OUTPUT_VIDEO_PATH)

Memuat model YOLO...
Model YOLO berhasil dimuat.

INFO: Customer ID 2 butuh bantuan (GESTUR INSTAN: raising-hand).
Processing frame 192 | Video time: 8.00s
Proses selesai. Video disimpan di: TES_ASSISTANCE GESTURE ALERT/sample7_output.mp4


In [13]:
if __name__ == "__main__":
    print("Memuat model YOLO...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    print("Model YOLO berhasil dimuat.")

    INPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample8.mp4'
    OUTPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample8_output.mp4'
    gesture_model = None

    process_video(yolo_model, gesture_model, INPUT_VIDEO_PATH, OUTPUT_VIDEO_PATH)

Memuat model YOLO...
Model YOLO berhasil dimuat.
Processing frame 192 | Video time: 8.00s
Proses selesai. Video disimpan di: TES_ASSISTANCE GESTURE ALERT/sample8_output.mp4


In [14]:
if __name__ == "__main__":
    print("Memuat model YOLO...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    print("Model YOLO berhasil dimuat.")

    INPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample9.mp4'
    OUTPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample9_output.mp4'
    gesture_model = None

    process_video(yolo_model, gesture_model, INPUT_VIDEO_PATH, OUTPUT_VIDEO_PATH)

Memuat model YOLO...
Model YOLO berhasil dimuat.
Processing frame 90 | Video time: 3.75s
INFO: Customer ID 2 butuh bantuan (GESTUR INSTAN: raising-hand).
Processing frame 157 | Video time: 6.54s
INFO: Customer ID 1 butuh bantuan (diam terlalu lama).
Processing frame 192 | Video time: 8.00s
Proses selesai. Video disimpan di: TES_ASSISTANCE GESTURE ALERT/sample9_output.mp4


In [15]:
if __name__ == "__main__":
    print("Memuat model YOLO...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    print("Model YOLO berhasil dimuat.")

    INPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample10.mp4'
    OUTPUT_VIDEO_PATH = 'TES_ASSISTANCE GESTURE ALERT/sample10_output.mp4'
    gesture_model = None

    process_video(yolo_model, gesture_model, INPUT_VIDEO_PATH, OUTPUT_VIDEO_PATH)

Memuat model YOLO...
Model YOLO berhasil dimuat.
Processing frame 140 | Video time: 5.83s
INFO: Customer ID 1 butuh bantuan (diam terlalu lama).
Processing frame 192 | Video time: 8.00s
Proses selesai. Video disimpan di: TES_ASSISTANCE GESTURE ALERT/sample10_output.mp4
